## Define the Docking Box
This notebook helps you to define the box dimensions needed for docking.

In [1]:
import nglview as nv
import numpy as np
from Bio.PDB import MMCIFParser, PDBParser

Insert your `cif` file:

In [2]:
input_file = "HIV1protease.pdb"
base, extension = input_file.split(".")

First we want to have an initial guess on where to put the box. To do this we take the center of the coordinates in the `.cif` file:

In [3]:
# Read the actual Cartesian coordinates from the CIF
if extension == "cif":
    structure = MMCIFParser(QUIET=True).get_structure("cif", input_file)
elif extension == "pdb":
    structure = PDBParser(QUIET=True).get_structure("pdb", input_file)

coords = np.array([atom.coord for atom in structure.get_atoms()])

# Use the geometric center of the CIF structure
coords_center = (coords.min(axis=0) + coords.max(axis=0)) / 2
print(f"Center of the input structure: {coords_center}")

Center of the input structure: [15.5494995 22.9005     3.7090006]


Next we want to visualize the target protein and the box. Insert the coordinates of the initial guess into `box_center` and start from there. You might execute the cell below multiple times with different box centers and lengths.

Note that the optimal box size depends on the size of the pocket and the size of the potential ligands that need to be docked into the pocket. Check out this [reference](https://pmc.ncbi.nlm.nih.gov/articles/PMC4468813/).

In [4]:
view = nv.show_file(input_file)
view.clear()
view.add_representation("cartoon", selection="protein")
view.add_representation("licorice", selection="water", opacity=0.1)

# Docking box
box_center = [15, 20, 10]  # x, y, z
box_lengths = [20, 20, 20]  # lx, ly, lz

### Do not modify beyond this point ###
c = np.array(box_center)
h = np.array(box_lengths) / 2

v = [
    c + s * h
    for s in [
        (-1, -1, -1),
        (1, -1, -1),
        (1, 1, -1),
        (-1, 1, -1),
        (-1, -1, 1),
        (1, -1, 1),
        (1, 1, 1),
        (-1, 1, 1),
    ]
]

view._add_shape(
    [
        ("cylinder", v[i], v[j], [0, 0, 0], 0.35)
        for i, j in [
            (0, 1),
            (1, 2),
            (2, 3),
            (3, 0),
            (4, 5),
            (5, 6),
            (6, 7),
            (7, 4),
            (0, 4),
            (1, 5),
            (2, 6),
            (3, 7),
        ]
    ]
)


view

NGLWidget()

Execute the final cell to see the results for the box dimensions.

In [5]:
print(f"Docking box center:  {box_center}")
print(f"Docking box lengths: {box_lengths}")

Docking box center:  [15, 20, 10]
Docking box lengths: [20, 20, 20]


Now we need to write the box dimensions into a file that our docking program can use:

In [6]:
with open("box_dimensions.txt", "w") as f:
    f.write(f"center_x = {box_center[0]:.3f}\n")
    f.write(f"center_y = {box_center[1]:.3f}\n")
    f.write(f"center_z = {box_center[2]:.3f}\n")
    f.write(f"size_x = {box_lengths[0]:.3f}\n")
    f.write(f"size_y = {box_lengths[1]:.3f}\n")
    f.write(f"size_z = {box_lengths[2]:.3f}\n")